# PD vs TBI Classification using Speech Analysis
## Smart and Connected Health - Complete ML Pipeline

This notebook implements the full pipeline:
1. **Feature Extraction** - 50+ speech features from audio files
2. **Feature Analysis** - PD vs TBI statistical comparison
3. **Model Training** - Random Forest, SVM, Gradient Boosting, Logistic Regression
4. **Evaluation** - Accuracy, Precision, Recall, F1, Confusion Matrix, ROC
5. **Inference** - Predict PD vs TBI from new audio

---
## Cell 1: Install Dependencies

In [2]:
# !pip install librosa praat-parselmouth scikit-learn xgboost matplotlib seaborn joblib -q
# !sudo apt-get install -y ffmpeg -qq

## Cell 2: Import Libraries

In [3]:
!pip install praat-parselmouth -q

import librosa
import parselmouth
from parselmouth.praat import call
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)
from sklearn.feature_selection import SelectKBest, f_classif

warnings.filterwarnings('ignore')
print('All libraries imported successfully!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 74.7 MB/s eta 0:00:00
All libraries imported successfully!


## Cell 3: Mount Google Drive (if using Drive for data)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## PART 1: ENHANCED FEATURE EXTRACTION (50+ Features)

In [5]:

from pathlib import Path
import logging
import shutil
import subprocess
from typing import Dict, Optional

output_dir = Path("tbi_audio_input_wav")
AUDIO_EXTENSIONS = (".wav", ".m4a", ".mp4", ".mp3")
TARGET_SR = 16000
LOGGER_NAME = "pd_tbi_pipeline"

logger = logging.getLogger(LOGGER_NAME)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)


class AudioProcessingError(Exception):
    """Raised when audio conversion or loading fails."""


class FeatureExtractionError(Exception):
    """Raised when feature extraction fails for a file."""


def ensure_ffmpeg_available() -> None:
    """Verify ffmpeg is installed and visible on PATH."""
    if shutil.which("ffmpeg") is None:
        raise EnvironmentError(
            "ffmpeg is not installed or not on PATH. "
            "Install ffmpeg before processing .m4a/.mp3/.mp4 files."
        )


def validate_audio_path(file_path: str | Path) -> Path:
    """Validate that a supported audio file exists."""
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Audio file not found: {path}")
    if path.suffix.lower() not in AUDIO_EXTENSIONS:
        raise ValueError(
            f"Unsupported audio format '{path.suffix}'. Supported formats: {AUDIO_EXTENSIONS}"
        )
    return path


def safe_float(value, default=np.nan) -> float:
    """Safely coerce a value to float."""
    try:
        if value is None:
            return default
        value = float(value)
        if np.isnan(value) or np.isinf(value):
            return default
        return value
    except Exception:
        return default


def _clean_array(values) -> np.ndarray:
    """Convert arbitrary numeric-like input to a clean 1D float array."""
    arr = np.asarray(values, dtype=float).reshape(-1)
    arr = arr[np.isfinite(arr)]
    return arr


def _distribution_stats(values, prefix: str, include_iqr: bool = False) -> Dict[str, float]:
    """Return a compact but rich summary for a 1D numeric vector."""
    arr = _clean_array(values)
    stats = {
        f"{prefix}_mean": np.nan,
        f"{prefix}_std": np.nan,
        f"{prefix}_median": np.nan,
        f"{prefix}_min": np.nan,
        f"{prefix}_max": np.nan,
    }
    if include_iqr:
        stats.update(
            {
                f"{prefix}_q25": np.nan,
                f"{prefix}_q75": np.nan,
                f"{prefix}_iqr": np.nan,
            }
        )

    if arr.size == 0:
        return stats

    stats.update(
        {
            f"{prefix}_mean": safe_float(np.mean(arr)),
            f"{prefix}_std": safe_float(np.std(arr)),
            f"{prefix}_median": safe_float(np.median(arr)),
            f"{prefix}_min": safe_float(np.min(arr)),
            f"{prefix}_max": safe_float(np.max(arr)),
        }
    )
    if include_iqr:
        q25, q75 = np.percentile(arr, [25, 75])
        stats.update(
            {
                f"{prefix}_q25": safe_float(q25),
                f"{prefix}_q75": safe_float(q75),
                f"{prefix}_iqr": safe_float(q75 - q25),
            }
        )

    return stats


def _matrix_row_stats(matrix, prefix: str, include_iqr: bool = False, include_global: bool = False) -> Dict[str, float]:
    """
    Summarize each row (e.g., coefficient/band) of a 2D feature matrix across time.
    Produces per-row stats and global stats.
    """
    x = np.asarray(matrix, dtype=float)
    if x.ndim == 1:
        x = x[np.newaxis, :]

    features = {}
    for idx, row in enumerate(x, start=1):
        row_prefix = f"{prefix}_{idx}"
        features[f"{row_prefix}_mean"] = safe_float(np.mean(row))
        features[f"{row_prefix}_std"] = safe_float(np.std(row))
        features[f"{row_prefix}_median"] = safe_float(np.median(row))
        features[f"{row_prefix}_min"] = safe_float(np.min(row))
        features[f"{row_prefix}_max"] = safe_float(np.max(row))
        if include_iqr:
            q25, q75 = np.percentile(row, [25, 75])
            features[f"{row_prefix}_q25"] = safe_float(q25)
            features[f"{row_prefix}_q75"] = safe_float(q75)
            features[f"{row_prefix}_iqr"] = safe_float(q75 - q25)

    if include_global:
        features.update(
            _distribution_stats(
                x.ravel(),
                f"{prefix}_global",
                include_iqr=include_iqr,
            )
        )
    return features


def _safe_feature_block(block_name: str, extractor_fn) -> Dict[str, float]:
    """
    Run an extractor block and continue even if it fails.
    This keeps batch processing robust while still logging the exact failing block.
    """
    try:
        return extractor_fn()
    except Exception as exc:
        logger.warning("%s failed: %s", block_name, exc)
        return {}


def convert_to_wav(input_path: str | Path, output_dir: Optional[str | Path] = output_dir) -> Path:
    """
    Convert m4a/mp4/mp3 to 16kHz mono wav. Returns the wav path.
    Existing wav files are returned as-is.
    """
    src = validate_audio_path(input_path)

    if src.suffix.lower() == ".wav":
        return src

    ensure_ffmpeg_available()

    out_dir = Path(output_dir) if output_dir is not None else src.parent
    out_dir.mkdir(parents=True, exist_ok=True)
    output_path = out_dir / f"{src.stem}.wav"

    cmd = [
        "ffmpeg",
        "-y",
        "-i", str(src),
        "-ar", str(TARGET_SR),
        "-ac", "1",
        str(output_path),
        "-loglevel", "error",
    ]

    logger.info("Converting %s -> %s", src.name, output_path.name)

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=False)
        if result.returncode != 0:
            raise AudioProcessingError(
                f"ffmpeg conversion failed for {src.name}: {result.stderr.strip() or 'unknown ffmpeg error'}"
            )
        if not output_path.exists():
            raise AudioProcessingError(f"Expected wav output was not created: {output_path}")
        return output_path
    except Exception as exc:
        raise AudioProcessingError(f"Audio conversion failed for {src}: {exc}") from exc


def extract_voice_quality_features(sound) -> Dict[str, float]:
    """
    Extract a comprehensive set of Parselmouth/Praat features:
    pitch, jitter, shimmer, HNR, intensity, formants, and spectral moments.
    """
    features = {}

    # Pitch
    pitch = call(sound, "To Pitch", 0.0, 75, 500)
    pitch_values = pitch.selected_array["frequency"]
    pitch_values = pitch_values[pitch_values > 0]
    features.update(_distribution_stats(pitch_values, "pitch_hz"))
    features["pitch_range"] = safe_float(
        features.get("pitch_hz_max", np.nan) - features.get("pitch_hz_min", np.nan)
    )
    pitch_diff = np.diff(pitch_values)
    features.update(_distribution_stats(pitch_diff, "pitch_diff"))
    total_frames = len(pitch.selected_array["frequency"])
    voiced_frames = len(pitch_values)
    features["voicing_ratio"] = safe_float(voiced_frames / total_frames if total_frames > 0 else 0)

    # Point process / perturbation features
    point_process = call(sound, "To PointProcess (periodic, cc)", 75, 500)
    features["jitter_local"] = safe_float(call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3))
    features["jitter_rap"] = safe_float(call(point_process, "Get jitter (rap)", 0, 0, 0.0001, 0.02, 1.3))
    features["jitter_ppq5"] = safe_float(call(point_process, "Get jitter (ppq5)", 0, 0, 0.0001, 0.02, 1.3))

    features["shimmer_local"] = safe_float(call([sound, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6))
    features["shimmer_apq3"] = safe_float(call([sound, point_process], "Get shimmer (apq3)", 0, 0, 0.0001, 0.02, 1.3, 1.6))
    features["shimmer_apq5"] = safe_float(call([sound, point_process], "Get shimmer (apq5)", 0, 0, 0.0001, 0.02, 1.3, 1.6))

    # Harmonicity / HNR
    harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
    try:
        harmonicity_values = harmonicity.values.flatten()
    except Exception:
        harmonicity_values = np.array([safe_float(call(harmonicity, "Get mean", 0, 0))])
    features.update(_distribution_stats(harmonicity_values, "hnr"))

    # Intensity
    intensity = call(sound, "To Intensity", 75.0, 0.0, "yes")
    try:
        intensity_values = intensity.values.flatten()
    except Exception:
        intensity_values = np.array([safe_float(call(intensity, "Get mean", 0, 0, "energy"))])
    features.update(_distribution_stats(intensity_values, "intensity_db"))
    features["intensity_range"] = safe_float(
        features.get("intensity_db_max", np.nan) - features.get("intensity_db_min", np.nan)
    )

    # Formants sampled across time
    formant = call(sound, "To Formant (burg)", 0.0, 5, 5500, 0.025, 50)
    times = np.arange(sound.xmin, sound.xmax, 0.01)
    for formant_idx in range(1, 5):
        values = [
            safe_float(call(formant, "Get value at time", formant_idx, float(t), "Hertz", "Linear"))
            for t in times
        ]
        # bandwidths = [
        #     safe_float(call(formant, "Get bandwidth at time", formant_idx, float(t), "Hertz", "Linear"))
        #     for t in times
        # ]
        features.update(_distribution_stats(values, f"formant_f{formant_idx}"))
        # features.update(_distribution_stats(bandwidths, f"formant_bw{formant_idx}"))

    return features


def extract_librosa_features(audio: np.ndarray, sr: int) -> Dict[str, float]:
    """
    Extract a broad set of librosa descriptors and summarize each feature channel over time.
    This is designed to produce 200+ stable features without changing downstream code.
    """
    features = {}

    # Shared intermediates
    stft = np.abs(librosa.stft(audio))
    power_spectrogram = stft ** 2
    harmonic_audio = librosa.effects.harmonic(audio)
    onset_env = librosa.onset.onset_strength(y=audio, sr=sr)

    # Cepstral features
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    delta_mfcc = librosa.feature.delta(mfcc)
    features.update(_matrix_row_stats(mfcc, "mfcc", include_iqr=False))
    features.update(_matrix_row_stats(delta_mfcc, "delta_mfcc", include_iqr=False))

    chroma_stft = librosa.feature.chroma_stft(S=power_spectrogram, sr=sr)
    spectral_contrast = librosa.feature.spectral_contrast(S=power_spectrogram, sr=sr)

    features.update(_distribution_stats(chroma_stft.flatten(), "chroma_stft", include_iqr=False))
    features.update(_distribution_stats(spectral_contrast.flatten(), "spectral_contrast", include_iqr=False))

    delta2_mfcc = librosa.feature.delta(mfcc, order=2)
    features.update(_matrix_row_stats(delta2_mfcc, "delta2_mfcc"))

    # Framewise scalar descriptors
    framewise_features = {
        "spectral_centroid": librosa.feature.spectral_centroid(S=stft, sr=sr).flatten(),
        "spectral_bandwidth_p2": librosa.feature.spectral_bandwidth(S=stft, sr=sr, p=2).flatten(),
        "spectral_flatness": librosa.feature.spectral_flatness(S=power_spectrogram).flatten(),
        "rms_energy": librosa.feature.rms(S=stft).flatten(),
        "zcr": librosa.feature.zero_crossing_rate(audio).flatten(),
    }
    for name, values in framewise_features.items():
        features.update(_distribution_stats(values, name))

    # Spectral rolloff
    rolloff = librosa.feature.spectral_rolloff(S=stft, sr=sr).flatten()
    features.update(_distribution_stats(rolloff, "spectral_rolloff"))

    # Spectral flux
    flux = np.sqrt(np.sum(np.diff(stft, axis=1)**2, axis=0))
    features.update(_distribution_stats(flux, "spectral_flux"))

    # Spectral entropy
    S_norm = stft / (np.sum(stft, axis=0, keepdims=True) + 1e-10)
    entropy = -np.sum(S_norm * np.log(S_norm + 1e-10), axis=0)
    features.update(_distribution_stats(entropy, "spectral_entropy"))

    # Tonnetz
    tonnetz = librosa.feature.tonnetz(y=harmonic_audio, sr=sr)
    features.update(_matrix_row_stats(tonnetz, "tonnetz"))

    # Rhythm features
    tempo = librosa.feature.tempo(onset_envelope=onset_env, sr=sr)
    features["tempo_bpm"] = safe_float(np.median(tempo))

    # Temporal / pause measures
    duration = librosa.get_duration(y=audio, sr=sr)
    non_silent = librosa.effects.split(audio, top_db=25)
    speech_segments = [(start / sr, end / sr) for start, end in non_silent]
    speech_durations = [end - start for start, end in speech_segments]
    pause_durations = [
        speech_segments[i + 1][0] - speech_segments[i][1]
        for i in range(len(speech_segments) - 1)
    ] if len(speech_segments) > 1 else []

    total_speech_time = float(np.sum(speech_durations)) if speech_durations else 0.0
    total_pause_time = max(0.0, duration - total_speech_time)

    features["duration_sec"] = safe_float(duration)
    features["num_pauses"] = safe_float(len(pause_durations), default=0.0)
    features["pause_fraction"] = safe_float(total_pause_time / duration if duration > 0 else 0.0, default=0.0)

    return features


def extract_all_features(file_path: str | Path, keep_converted: bool = True) -> Dict[str, float]:
    """
    Extract a large combined feature set from a single file with clear failure points.
    The extractor is block-based: if one block fails, the rest continue, and only a
    completely empty result raises an exception.
    """
    src = validate_audio_path(file_path)
    temporary_wav = False

    try:
        if src.suffix.lower() in {".m4a", ".mp4", ".mp3"}:
            wav_path = convert_to_wav(src, output_dir)
            temporary_wav = not keep_converted and wav_path.parent == Path(output_dir)
        else:
            wav_path = src

        audio, sr = librosa.load(str(wav_path), sr=TARGET_SR, mono=True)
        if audio is None or len(audio) == 0:
            raise AudioProcessingError(f"Loaded audio is empty: {wav_path}")

        sound = parselmouth.Sound(str(wav_path))

        features = {}
        features.update(_safe_feature_block("parselmouth_voice_quality", lambda: extract_voice_quality_features(sound)))
        features.update(_safe_feature_block("librosa_feature_bank", lambda: extract_librosa_features(audio, sr)))

        if not features:
            raise FeatureExtractionError(f"No features were extracted from {src.name}")

        features["total_feature_count"] = safe_float(len(features), default=0.0)
        return features

    except Exception as exc:
        raise FeatureExtractionError(f"Failed to extract features from {src.name}: {exc}") from exc

    finally:
        if temporary_wav:
            try:
                Path(wav_path).unlink(missing_ok=True)
            except Exception as cleanup_exc:
                logger.warning("Could not remove temporary wav %s: %s", wav_path, cleanup_exc)


print("Comprehensive feature extraction helpers defined.")
# print("This version extracts a much larger librosa + Parselmouth feature bank (200+ features) with block-level failure isolation.")


Comprehensive feature extraction helpers defined.


---
## PART 2: Test Feature Extraction on a Single File

In [6]:
# Upload your audio file or use a Drive path
# Option A: Upload directly
# from google.colab import files
# uploaded = files.upload()

# Option B: Use Drive path
test_file = "/content/drive/MyDrive/tbi_audio_input/tbi_audio_input/20250319_141117_HBOT_070_Grandfather.m4a"

try:
    features = extract_all_features(test_file)
    print("=" * 60)
    print("EXTRACTED FEATURES")
    print("=" * 60)
    print(f"{'Feature':<30} {'Value':>15}")
    print("-" * 47)
    for name, value in features.items():
        if isinstance(value, (float, int, np.floating, np.integer)):
            print(f"  {name:<28} {float(value):>15.6f}")
        else:
            print(f"  {name:<28} {str(value):>15}")
    print("-" * 47)
    print(f"Total features: {len(features)}")
except Exception as exc:
    logger.exception("Single-file feature extraction failed for %s", test_file)
    print(f"[ERROR] Could not extract features from {test_file}: {exc}")

[INFO] Converting 20250319_141117_HBOT_070_Grandfather.m4a -> 20250319_141117_HBOT_070_Grandfather.wav
INFO:pd_tbi_pipeline:Converting 20250319_141117_HBOT_070_Grandfather.m4a -> 20250319_141117_HBOT_070_Grandfather.wav


EXTRACTED FEATURES
Feature                                  Value
-----------------------------------------------
  pitch_hz_mean                     119.676363
  pitch_hz_std                       88.426560
  pitch_hz_median                    93.646314
  pitch_hz_min                       74.951181
  pitch_hz_max                      499.450820
  pitch_range                       424.499639
  pitch_diff_mean                     0.163723
  pitch_diff_std                     49.528671
  pitch_diff_median                  -0.141662
  pitch_diff_min                   -408.481624
  pitch_diff_max                    413.062004
  voicing_ratio                       0.415856
  jitter_local                        0.031275
  jitter_rap                          0.014518
  jitter_ppq5                         0.016360
  shimmer_local                       0.166973
  shimmer_apq3                        0.075326
  shimmer_apq5                        0.108882
  hnr_mean                         -126.

---
## PART 3: Batch Process All Audio Files (PD & TBI Folders)

In [8]:
import os
import pandas as pd
import re
from dataclasses import dataclass, asdict
from datetime import datetime

@dataclass
class ProcessingSummary:
    total_seen: int = 0
    processed_ok: int = 0
    failed: int = 0
    skipped_unsupported: int = 0

def parse_filename(filename):
    """
    Expected pattern:
    20250319_141117_HBOT_070_Grandfather.m4a
    Returns: label, patient_id, name, date, time
    """
    pattern = r"^(\d{8})_(\d{6})_([A-Za-z]+)_(\d+)_([^.]+)\.[^.]+$"
    match = re.match(pattern, filename)
    if not match:
        logger.warning("Filename did not match expected pattern: %s", filename)
        return "", "", "", "", ""

    date, time, group, patient_id, name = match.groups()
    label = "TBI" if group.upper() == "HBOT" else "PD"
    return label, patient_id, name, date, time

def process_folder(folder_path, label, save_errors_to="processing_errors.csv"):
    """Process all supported audio files in a folder and capture failures for debugging."""
    folder = Path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder.resolve()}")
    if not folder.is_dir():
        raise NotADirectoryError(f"Expected a directory, got: {folder.resolve()}")

    rows = []
    error_rows = []
    summary = ProcessingSummary()

    logger.info("Processing folder: %s", folder.resolve())

    for file_path in sorted(folder.iterdir()):
        summary.total_seen += 1

        if not file_path.is_file():
            logger.debug("Skipping non-file entry: %s", file_path)
            continue

        if file_path.suffix.lower() not in AUDIO_EXTENSIONS:
            summary.skipped_unsupported += 1
            logger.debug("Skipping unsupported extension: %s", file_path.name)
            continue

        parsed_label, patient_id, name, date, time = parse_filename(file_path.name)

        try:
            feat = extract_all_features(file_path)
            row = {
                "label": parsed_label or label,
                "patient_id": patient_id,
                "file_name": name or file_path.stem,
                "date": date,
                "time": time,
                "audio_path": str(file_path),
            }
            row.update(feat)
            rows.append(row)
            summary.processed_ok += 1
            logger.info("[OK] %s", file_path.name)

        except Exception as exc:
            summary.failed += 1
            logger.exception("Failed processing %s", file_path.name)
            error_rows.append(
                {
                    "timestamp": datetime.now().isoformat(timespec="seconds"),
                    "folder": str(folder),
                    "file_name": file_path.name,
                    "audio_path": str(file_path),
                    "default_label": label,
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                }
            )
            print(f"[FAIL] {file_path.name}: {exc}")

    if error_rows:
        error_df = pd.DataFrame(error_rows)
        error_df.to_csv(save_errors_to, index=False)
        print(f"\nSaved error log to {save_errors_to}")

    return rows, summary, error_rows

# ===== SET YOUR DATA PATHS HERE =====
TBI_FOLDER =  "/content/drive/MyDrive/tbi_audio_input/tbi_audio_input"
PD_FOLDER = "pd_audio_input"

all_rows = []
all_errors = []

print("Processing TBI files...")
tbi_rows, tbi_summary, tbi_errors = process_folder(TBI_FOLDER, "TBI", save_errors_to="tbi_processing_errors.csv")
all_rows.extend(tbi_rows)
all_errors.extend(tbi_errors)

pd_rows, pd_summary = [], ProcessingSummary()
if Path(PD_FOLDER).exists() and any(Path(PD_FOLDER).iterdir()):
    print("\nProcessing PD files...")
    pd_rows, pd_summary, pd_errors = process_folder(PD_FOLDER, "PD", save_errors_to="pd_processing_errors.csv")
    all_rows.extend(pd_rows)
    all_errors.extend(pd_errors)
else:
    print("\nPD folder not found or empty. Skipping PD processing.")

df = pd.DataFrame(all_rows)

metadata_cols = ["label", "patient_id", "file_name", "date", "time", "audio_path"]
feature_count = max(0, len(df.columns) - len(metadata_cols))

print("\n" + "=" * 60)
print("BATCH PROCESSING SUMMARY")
print("=" * 60)
print(f"TBI processed: {tbi_summary.processed_ok}/{tbi_summary.total_seen}")
print(f"PD processed : {pd_summary.processed_ok}/{pd_summary.total_seen}")
print(f"Failures     : {tbi_summary.failed + pd_summary.failed}")
print(f"Saved rows   : {len(df)}")
print(f"Features/file: {feature_count}")

if not df.empty:
    df.to_csv("features.csv", index=False)
    print("\nSaved features to features.csv")
else:
    print("\n[WARNING] No features were extracted. features.csv was not written.")

if all_errors:
    pd.DataFrame(all_errors).to_csv("processing_errors_all.csv", index=False)
    print("Saved combined error log to processing_errors_all.csv")

df.head()

[INFO] Processing folder: /content/drive/MyDrive/tbi_audio_input/tbi_audio_input
INFO:pd_tbi_pipeline:Processing folder: /content/drive/MyDrive/tbi_audio_input/tbi_audio_input
[INFO] Converting 20241127_093959_HBOT_004_Grandfather.m4a -> 20241127_093959_HBOT_004_Grandfather.wav
INFO:pd_tbi_pipeline:Converting 20241127_093959_HBOT_004_Grandfather.m4a -> 20241127_093959_HBOT_004_Grandfather.wav


Processing TBI files...


[INFO] [OK] 20241127_093959_HBOT_004_Grandfather.m4a
INFO:pd_tbi_pipeline:[OK] 20241127_093959_HBOT_004_Grandfather.m4a
[INFO] Converting 20241127_094136_HBOT_004_Grandfather.m4a -> 20241127_094136_HBOT_004_Grandfather.wav
INFO:pd_tbi_pipeline:Converting 20241127_094136_HBOT_004_Grandfather.m4a -> 20241127_094136_HBOT_004_Grandfather.wav
[INFO] [OK] 20241127_094136_HBOT_004_Grandfather.m4a
INFO:pd_tbi_pipeline:[OK] 20241127_094136_HBOT_004_Grandfather.m4a
[INFO] Converting 20241127_135232_HBOT_008_Grandfather.m4a -> 20241127_135232_HBOT_008_Grandfather.wav
INFO:pd_tbi_pipeline:Converting 20241127_135232_HBOT_008_Grandfather.m4a -> 20241127_135232_HBOT_008_Grandfather.wav
[INFO] [OK] 20241127_135232_HBOT_008_Grandfather.m4a
INFO:pd_tbi_pipeline:[OK] 20241127_135232_HBOT_008_Grandfather.m4a
[INFO] Converting 20241127_135344_HBOT_008_Grandfather.m4a -> 20241127_135344_HBOT_008_Grandfather.wav
INFO:pd_tbi_pipeline:Converting 20241127_135344_HBOT_008_Grandfather.m4a -> 20241127_135344_HBOT_


PD folder not found or empty. Skipping PD processing.

BATCH PROCESSING SUMMARY
TBI processed: 132/133
PD processed : 0/0
Failures     : 0
Saved rows   : 132
Features/file: 329

Saved features to features.csv


,label,patient_id,file_name,date,time,audio_path,pitch_hz_mean,pitch_hz_std,pitch_hz_median,pitch_hz_min,...,tonnetz_6_mean,tonnetz_6_std,tonnetz_6_median,tonnetz_6_min,tonnetz_6_max,tempo_bpm,duration_sec,num_pauses,pause_fraction,total_feature_count
0,TBI,004,Grandfather,20241127,093959,/content/drive/MyDrive/tbi_audio_input/tbi_aud...,105.497645,41.203214,98.232871,74.946068,...,0.000664,0.054881,-0.002378,-0.168635,0.244309,133.928571,84.224000,45.0,0.363982,328.0
1,TBI,004,Grandfather,20241127,094136,/content/drive/MyDrive/tbi_audio_input/tbi_aud...,108.506524,40.609695,100.560394,75.182121,...,0.000848,0.055562,0.000473,-0.180146,0.237168,110.294118,72.192000,46.0,0.333777,328.0
2,TBI,008,Grandfather,20241127,135232,/content/drive/MyDrive/tbi_audio_input/tbi_aud...,108.797925,28.813107,102.522123,75.383432,...,-0.002429,0.054493,-0.000285,-0.170925,0.238671,117.187500,70.229375,44.0,0.312424,328.0
3,TBI,008,Grandfather,20241127,135344,/content/drive/MyDrive/tbi_audio_input/tbi_aud...,105.255715,26.053848,102.473152,75.081619,...,-0.012027,0.050354,-0.010434,-0.153824,0.146590,125.000000,46.848000,27.0,0.199454,328.0
4,TBI,002,Grandfather,20241205,112453,/content/drive/MyDrive/tbi_audio_input/tbi_aud...,113.621797,58.450392,94.921148,75.353015,...,0.039924,0.055819,0.036132,-0.171067,0.255393,125.000000,49.408000,34.0,0.290803,328.0


---
## PART 4: Feature Analysis - PD vs TBI Comparison

In [ ]:
# Load features (or use df from above)
df = pd.read_csv('features.csv')
drop_cols = ['filename', 'label']
feature_cols = [c for c in df.columns if c not in drop_cols]

# Group statistics
print('PD vs TBI Feature Comparison')
print('=' * 75)

stats = df.groupby('label')[feature_cols].agg(['mean', 'std'])
print(f"\n{'Feature':<25} {'PD Mean':>12} {'PD Std':>12} {'TBI Mean':>12} {'TBI Std':>12}")
print('-' * 75)
for feat in feature_cols:
    pd_m = stats.loc['PD', (feat, 'mean')] if 'PD' in stats.index else 0
    pd_s = stats.loc['PD', (feat, 'std')] if 'PD' in stats.index else 0
    tbi_m = stats.loc['TBI', (feat, 'mean')] if 'TBI' in stats.index else 0
    tbi_s = stats.loc['TBI', (feat, 'std')] if 'TBI' in stats.index else 0
    print(f'  {feat:<23} {pd_m:>12.4f} {pd_s:>12.4f} {tbi_m:>12.4f} {tbi_s:>12.4f}')

In [ ]:
# Top discriminating features (ANOVA F-test)
X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(df[feature_cols].median())
le = LabelEncoder()
y = le.fit_transform(df['label'])

selector = SelectKBest(f_classif, k=min(15, len(feature_cols)))
selector.fit(X, y)

scores = pd.DataFrame({
    'feature': feature_cols,
    'f_score': selector.scores_,
    'p_value': selector.pvalues_
}).sort_values('f_score', ascending=False)

print('\nTop 15 Discriminating Features (ANOVA F-test)')
print('-' * 50)
for _, row in scores.head(15).iterrows():
    print(f"  {row['feature']:<25} F={row['f_score']:>10.4f}  p={row['p_value']:.6f}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
top = scores.head(15)
ax.barh(top['feature'][::-1], top['f_score'][::-1], color='steelblue')
ax.set_xlabel('F-Statistic')
ax.set_title('Top 15 Discriminating Features: PD vs TBI')
plt.tight_layout()
plt.savefig('feature_importance_analysis.png', dpi=150)
plt.show()

---
## PART 5: Model Training & Evaluation

In [ ]:
# Prepare data
df = pd.read_csv('features.csv')
drop_cols = ['filename', 'label']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(df[feature_cols].median())
le = LabelEncoder()
y = le.fit_transform(df['label'])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Dataset: {X_scaled.shape[0]} samples, {X_scaled.shape[1]} features')
print(f'Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')
print(f'Class distribution: {dict(zip(le.classes_, np.bincount(y)))}')

In [ ]:
# Define models
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=42, class_weight='balanced'
    ),
    'SVM (RBF)': SVC(
        kernel='rbf', C=1.0, gamma='scale', probability=True,
        class_weight='balanced', random_state=42
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42
    ),
}

# Cross-validation
n_splits = min(5, min(np.bincount(y)))
n_splits = max(2, n_splits)
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

print(f'{n_splits}-Fold Stratified Cross-Validation')
print('=' * 60)

results = {}
for name, model in models.items():
    cv_acc = cross_val_score(model, X_scaled, y, cv=skf, scoring='accuracy')
    cv_f1 = cross_val_score(model, X_scaled, y, cv=skf, scoring='f1_weighted')
    results[name] = {'acc': cv_acc.mean(), 'f1': cv_f1.mean()}
    print(f'\n{name}:')
    print(f'  Accuracy: {cv_acc.mean():.4f} (+/- {cv_acc.std():.4f})')
    print(f'  F1 Score: {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})')

In [ ]:
# Detailed evaluation on train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print('Detailed Evaluation (80/20 Split)')
print('=' * 60)

best_f1 = -1
best_model = None
best_name = None

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    print(f'\n{name}: Accuracy={acc:.4f}, F1={f1:.4f}')
    print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_name = name

print(f'\nBest Model: {best_name} (F1={best_f1:.4f})')

In [ ]:
# Confusion Matrix
y_pred_best = best_model.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix - {best_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ROC Curve
if hasattr(best_model, 'predict_proba'):
    y_prob = best_model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.4f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve - {best_name}')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig('roc_curve.png', dpi=150)
    plt.show()

In [ ]:
# Feature Importance (tree-based models)
# Retrain best model on full data
best_model_full = models[best_name]
best_model_full.fit(X_scaled, y)

if hasattr(best_model_full, 'feature_importances_'):
    importances = best_model_full.feature_importances_
    feat_imp = pd.DataFrame({
        'feature': feature_cols,
        'importance': importances
    }).sort_values('importance', ascending=False)

    fig, ax = plt.subplots(figsize=(10, 8))
    top20 = feat_imp.head(20)
    ax.barh(top20['feature'][::-1], top20['importance'][::-1], color='teal')
    ax.set_xlabel('Importance')
    ax.set_title(f'Top 20 Feature Importances - {best_name}')
    plt.tight_layout()
    plt.savefig('feature_importances.png', dpi=150)
    plt.show()

---
## PART 6: Save Model for Inference

In [ ]:
# Save model artifacts
joblib.dump(best_model_full, 'best_model.joblib')
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(le, 'label_encoder.joblib')
joblib.dump(feature_cols, 'feature_names.joblib')

print('Model artifacts saved:')
print('  best_model.joblib')
print('  scaler.joblib')
print('  label_encoder.joblib')
print('  feature_names.joblib')

---
## PART 7: Inference - Predict from New Audio File

In [ ]:
def predict_from_audio(file_path):
    """Predict PD vs TBI from a new audio file."""
    # Load model artifacts
    model = joblib.load('best_model.joblib')
    scaler = joblib.load('scaler.joblib')
    le = joblib.load('label_encoder.joblib')
    feature_names = joblib.load('feature_names.joblib')

    # Extract features
    print(f'Extracting features from: {file_path}')
    features = extract_all_features(file_path)

    # Build feature vector in correct order
    feature_vector = []
    for fname in feature_names:
        val = features.get(fname, 0.0)
        if val is None or (isinstance(val, float) and (np.isnan(val) or np.isinf(val))):
            val = 0.0
        feature_vector.append(val)

    feature_vector = np.array(feature_vector).reshape(1, -1)
    feature_vector_scaled = scaler.transform(feature_vector)

    # Predict
    prediction = model.predict(feature_vector_scaled)[0]
    predicted_label = le.inverse_transform([prediction])[0]

    if hasattr(model, 'predict_proba'):
        probas = model.predict_proba(feature_vector_scaled)[0]
        confidence = probas.max()
        class_probas = dict(zip(le.classes_, probas))
    else:
        confidence = None
        class_probas = {}

    print('\n' + '=' * 50)
    print('PREDICTION RESULT')
    print('=' * 50)
    print(f'  File:       {os.path.basename(file_path)}')
    print(f'  Prediction: {predicted_label}')
    if confidence is not None:
        print(f'  Confidence: {confidence:.2%}')
        for cls, prob in class_probas.items():
            print(f'    P({cls}): {prob:.4f}')
    print('=' * 50)

    return predicted_label, confidence, class_probas


# Test inference
# predict_from_audio('/content/new_patient_recording.m4a')

---
## Summary

### Features Extracted (54 total)
| Category | Count | Features |
|----------|-------|----------|
| Voice Quality (Parselmouth) | 15 | Pitch (4), Jitter (3), Shimmer (3), HNR (1), Formants (4) |
| Spectral (Librosa) | 32 | MFCCs (13), Delta MFCCs (13), Centroid, Bandwidth, Rolloff, Flatness, Contrast, Chroma |
| Temporal (Librosa) | 7 | Duration, ZCR, RMS Energy, Pauses (2), Speech Rate, Tempo |

### Models Trained
| Model | Description |
|-------|-------------|
| Random Forest | Ensemble of decision trees, robust to overfitting |
| SVM (RBF) | Support Vector Machine with radial basis function kernel |
| Gradient Boosting | Sequential ensemble, strong on tabular data |
| Logistic Regression | Linear baseline, interpretable |

### Evaluation Metrics
- Stratified K-Fold Cross-Validation
- Accuracy, Precision, Recall, F1 Score
- Confusion Matrix, ROC Curve, AUC